In [1]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain import hub
from langchain.tools import tool 
import requests

c:\Users\devch\anaconda3\envs\langagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_react_agent, AgentExecutor # reasoning and acting agent

In [3]:
# load environment variables from .env file

os.environ["SSL_CERT_FILE"] = certifi.where() # set the SSL certificate file path for secure connections
# why do we need to set the SSL certificate file path?
# Setting the SSL certificate file path is necessary to ensure that the application can establish secure connections when making API calls or accessing resources over HTTPS. The SSL certificate file contains trusted certificates that allow the application to verify the identity of the server it is connecting to, ensuring that the communication is secure
load_dotenv() # load environment variables from .env file

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") 
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_STACK_API_KEY = os.getenv("WEATHER_STACK_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=3, api_key=TAVILY_API_KEY)

In [5]:
# custom function to get the weather information
@tool
def get_weather_data(location: str) -> str:

    '''
    This function takes a location as input and returns the current weather information for that location using the Weatherstack API.
    '''

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_STACK_API_KEY}&query={location}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not retrieve weather data for {location}."
    
    return (
        f"City: {location}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
# result = search_tool.invoke("What is the latest news about AI?")
# result

In [7]:
# Intializing our large language model (LLM) with the Gemini API key

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=GEMINI_API_KEY, 
    temperature=0.2
)


In [8]:
# result = llm.invoke("What year is it?")
# result

In [9]:
from langsmith import Client

# prompt for the agent to use, we can pull it from the LangSmith Hub

client = Client()

prompt = client.pull_prompt("hwchase17/react")

prompt

c:\Users\devch\anaconda3\envs\langagent\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [10]:
# tools

tools = [search_tool, get_weather_data]

In [11]:
# Create the agent using the LLM, tools, and prompt

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt 
)


In [15]:
# Executer 

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    intermediate_steps=True,
    handle_parsing_errors=True
)

In [16]:
# Run

response = agent_executor.invoke({
    "input": (
        "Current weather in New York and the latest news about AI."
    )
})



> Entering new AgentExecutor chain...
Action: get_weather_data
Action Input: New YorkCity: New York
Temperature: 19°C
Weather: Sunny
Humidity: 68%Action: tavily_search_results_json
Action Input: latest news about AI[{'url': 'https://www.sciencedaily.com/news/computers_math/artificial_intelligence', 'content': 'ScienceDaily\n\n# Artificial Intelligence News\n\n## Top Headlines\n\n## Latest Headlines\n\n## Earlier Headlines\n\n### Friday, March 13, 2026\n\n### Friday, February 20, 2026\n\n### Tuesday, February 10, 2026\n\n### Friday, February 6, 2026\n\n### Saturday, January 31, 2026\n\n### Thursday, January 29, 2026\n\n### Monday, February 2, 2026\n\n### Wednesday, January 28, 2026\n\n### Sunday, January 25, 2026\n\n### Friday, January 16, 2026\n\n### Tuesday, January 20, 2026\n\n### Saturday, February 14, 2026\n\n### Friday, January 9, 2026\n\n### Wednesday, January 7, 2026\n\n### Monday, December 22, 2025\n\n### Wednesday, December 31, 2025\n\n### Tuesday, January 6, 2026\n\n### Fri

In [14]:
print(response["output"])

The current weather in New York is 19°C and Clear with 65% humidity. Latest AI news includes Aviva deploying AI to stop £230M in insurance fraud, Google folding Display Ads into an AI-first Demand Gen platform, insurers pivoting AI strategy toward core risk underwriting, Visa ChatGPT integration enabling AI agent retail purchasing, and McDonald's testing a Google-backed AI drive-thru ordering system. Other news covers Google Cloud generative AI automating council planning operations, the EU publishing its AI content labelling playbook, HarmonyOS 7 stepping into the AI gap Apple left open in China, and Accenture reporting growing consumer trust in AI shopping agents. There's also news about Meta Business Agent driving AI-powered conversational commerce, China's AI mapping its renewable energy grid, and IBM Research unveiling a breakthrough analog AI chip.
